### Programming for Science and Finance

*Prof. Götz Pfeiffer, School of Mathematical and Statistical Sciences, University of Galway*

# Notebook 7: Functions and Data

This notebook accompanies **Part II: Scientific Programming**. You will:

* understand how Python treats **functions as first-class objects**, and why this is useful
* see how **anonymous functions** and **closures** let a function capture and carry state
* pass an arbitrary number of arguments to a function, using `*args` and `**kwargs`
* use the `pandas` types **`Series`** and **`DataFrame`** to work with data that carries **labels**, and not just positions
* slice, difference and average a **time series** of stock prices, and meet the moving averages of Notebook 6 again in a new guise
* summarize and compare several series at once, through their **volatilities** and **correlations**

By the end of this notebook, you will be able to write clear, reusable and expressive Python code, and to apply it to data that arrives labelled — as real data usually does.

In [ ]:
import math
import numpy as np
import matplotlib.pyplot as plt

## Task 1: Function Attributes.

Functions in Python are **first class objects**.  
As such they can be assigned to variables,  passed as arguments into functions and also serve as return values.  
Like other objects, functions have **attributes** that can be accessed through the dot notation `obj.attr`.

**Example.** If we define a function `cube` with a **docstring** like this:

In [ ]:
def cube(x):
    """computes the cube of x"""
    return x * x * x

then the docstring becomes the function's `__doc__` attribute:

In [ ]:
cube.__doc__

and is reported by the built-in function `help`.  (In an interactive session, `?cube` does the same, in a separate window -- but, unlike `help`, it prints nothing into the notebook itself.)

In [ ]:
help(cube)

A function has a name:

In [ ]:
cube.__name__

Most importantly, a function implements the special method `__call__` which makes it **callable**.  In some sense, `cube(3)` is just a shorthand for:

In [ ]:
cube.__call__(3)

---
**Exercises.**

1. What is the `type` of a function object, like `cube`?

2. The builtin python function `dir` lists all the attributes of an object.  How many attributes does the function  `cube` have?

3. Assign `f = cube`, then look at `f.__name__`.  What does the answer say about the relation between a function and the name it happens to be stored under?

4. Being objects, functions can be collected in a list.  Put `cube`, `abs` and `len` into one, and call each of them on a suitable argument in a `for` loop.

---

## Task 2: Anonymous Functions.

Some functions don't need a name.  
An **anonymous function** can be defined by a **lambda expression** of the form
```
lambda params : expression
```
Here, `lambda` is a **keyword**, `params` is a list of names used as **parameters**, and `expression` is a Python **formula** for computing the output of the function.

Not every function can be written in this way.   But a function that is used only once and just computes the value of an expression is disposable and doesn't need a name.

In [ ]:
(lambda x : x**3)(3)

We might want to define a **function factory**, i.e., a function that makes and returns different functions, depending on an input parameter.

In [ ]:
def make_power(n):
    return lambda x : x**n

So, `make_power(2)` returns the function $x \mapsto x^2$, `make_power(3)` returns $x \mapsto x^3$, and so on.

In [ ]:
p3 = make_power(3)
p3(3)

We might want to define a function `shifted` that returns a shifted variant of a function `f`, shifted vertically by `v`, and horizontally by `h`. 

In [ ]:
def shifted(f, v, h):
    return lambda x: f(x - h) + v

This is an example of a function that has a function as input and returns a function as output.  
For example, suppose `f` is the function $f(x) = \frac12(x^3 - x + 1)$.

In [ ]:
f = lambda x: (x**3 - x + 1)/2

We can compute and plot $f(x)$  for $x \in [-1.8, 1.1]$, and the same graph shifted vertically by $2$ or by $-1$, and shifted horizontally by $\frac12$ or by $-\frac12$ as follows.

In [ ]:
xxx = np.linspace(-1.8, 1.1, 50)
params = [(0,0), (2,0), (-1,0), (0,1/2), (0,-1/2)]
for v, h in params:
    plt.plot(xxx+h, [shifted(f, v, h)(x) for x in xxx+h],
             label=f"v = {v}, h = {h}")
_ = plt.legend()

Each curve is the same parabola, moved `h` to the right and `v` upwards.  Note that `shifted` has not modified `f`: it has built five new functions, each of which remembers its own `v` and `h`.

### Closures.

How can the function returned by `make_power(3)` still know the value of `n`, when the call to `make_power` has long finished, and its local variables should be gone (Notebook 1)?

The answer is that a function defined **inside** another function keeps a reference to those variables of the enclosing function that it actually uses.  A function together with this captured environment is called a **closure**.  Each call of `make_power` builds a new closure, with its own value of `n`.

In [ ]:
p2 = make_power(2)
p3 = make_power(3)
print(p2(5), p3(5))   # 25 125: each lambda kept its own n
print(p3.__closure__[0].cell_contents)   # the captured value of n

A closure is what lets a function carry state without a class.  Here `p2` and `p3` each remember their own `n`, long after `make_power` has returned.  The same mechanism is behind every function factory: `shifted(f, v, h)` above returns a function that has captured `f`, `v` and `h`.

A closure can hold more than a number.  If it captures a **dictionary**, the returned
function has a private store that survives from one call to the next.  This is the idea
behind **memoization**: remember each result, and never compute it twice.

In [ ]:
def memoize(f):
    cache = {}                       # captured by the closure below
    def ff(n):
        if n not in cache:
            cache[n] = f(n)
        return cache[n]
    return ff

fib = memoize(lambda n: n if n < 2 else fib(n-2) + fib(n-1))
print(fib(30))                       # 832040, and instantly

Without the cache, `fib(30)` would call itself more than a million times, as each value
is recomputed from scratch in both branches of the recursion.  With it, each of the $31$
values is computed exactly once.  Python's standard library offers this as
`functools.cache`, applied with the `@` notation of Notebook 4.

---
**Exercises.**

1. Define a function `evaluate` that takes a function `f` and a list `xxx` of $x$-values as argument, and returns the list of values $f(x)$ for $x$ in `xxx`.

2. Following the pattern of `make_power`, write a function factory `make_poly(a, b, c)` that returns the quadratic function $x \mapsto a x^2 + b x + c$.  Use it to plot three different parabolas in one figure.

3. What does `p3.__closure__[0].cell_contents` report?  Call `make_power(4)` afterwards and look again.  Why is `p3` unaffected?

4. Extend `memoize` so that the function it returns also counts, in `ff.calls`, how often the underlying `f` was actually evaluated.  How many evaluations does `fib(30)` need with the cache, and how many would it need without?

---


## Task 3: Multiple Arguments.

Python uses a simple method to allow a function to have multiple arguments, without having to specify their exact number.

Recall **list unpacking**:  putting `*` in front of a list (or tuple), essentially removes
a pair of brackets (or parentheses), `*(1, 2, 3)` becomes `1, 2, 3` in a suitable context,
like the following:

In [ ]:
[0, *(1, 2, 3), 4]

In reverse, **list packing** adds a pair of brackets, when needed, for example in a parallel assignment:

In [ ]:
[a, *b, c] = [0, 1, 2, 3, 4]
print(f"a = {a}")
print(f"b = {b}")
print(f"c = {c}")

Both operations can be used at the same time:

In [ ]:
data = [2, 3, 4]
[x, y, *z] = [1, *data, 5]
print(f"x = {x}")
print(f"y = {y}")
print(f"z = {z}")

In a function definition, the **last** parameter can be a list packing parameter, often called `*args`.  
For context, let's briefly look at how the **arguments** given in a **function call** are matched up with the **parameters** declared in a **function definition**, either by their **position** or by a **keyword**.

### Function Call.

In Python, a typical function call has the form
```python
value = f(1, 2, 3, ..., e=2, d=2, ...)
```
Here the arguments `1, 2, 3, ...` are **positional** arguments: they are matched up by their position in the argument list, with the parameter at the same position in the function definition.  
In contrast, the arguments `e=2, d=2, ...` are **keyword** arguments:  they
are matched by the keyword with the parameter of the same name in the function definition.
(It therefore doesn't matter in which order they are listed.)  
In any case, the positional arguments must come before the keyword arguments: they cannot be mixed.

### Function Definition.

In Python, a typical function definition head has the form
```python
def f(a, b, c, ..., d=0, e=1, ...):
```
Here, the parameters `a, b, c, ...` are **required** parameters: values for them need to be provided as arguments in the function call.   
In contrast, the parameters `d, e, ...` are **optional** parameters equipped with **default values** `d=0, e=1, ...`. If the function call provides a value for an optional parameter, this value overrides the default value.
In any case, the required parameters must come before the optional parameters: they cannot be mixed.

A function definition can use argument unpacking to collect several positional arguments in a list `args`, and several keyword arguments in a dictionary `kwargs` by including their starred versions in the parameter list:
```python
def f(..., *args, ..., **kwargs):
```
If a function is defined in this way, the parameters before `*args` can only receive
positional arguments.  After matching these up, all the remaining positional arguments
are collected as a list in `args`.  The parameters between `*args` and `**kwargs` can only
receive keyword arguments.  After matching these up, all remaining keyword arguments are 
collected in a dictionary `kwargs`.

Here `args` and `kwargs` are just the names used by convention.

If applied correctly, this can be a powerful way to add flexibility to your function calls.

---
**Exercises.**

1. Some function definition heads contain the symbols `/` or `*` as explicit parameters,
   i.e., they look like

   ```python
   def f(..., /, ..., *, ...):
   ```

   Consult the documentation to find out what that means.

2.  Define a silly `demo` function as

    ```python
    def demo(a, b, /, c, *, d):
        return a, b, c, d
    ```

    Then try

    ```python
    demo(1, 2, 3, d=4)
    demo(a=1, b=2, c=3, d=4)
    demo(1,2,c=3,4)
    ```

    and explain the resulting error messages, if any.

3. Write a function `summary(*args, **kwargs)` whose body simply prints `args` and `kwargs`.  Call it with a mixture of positional and keyword arguments, and check that each one lands where you expect.

4. Write a function `total(*numbers)` that adds up any number of arguments, so that both `total(1, 2, 3)` and `total(*[1, 2, 3])` return $6$.

---

## Task 4: Series and DataFrames.

So far, all our data has been a `list` or a numpy `array`: bare numbers, identified by
their **position**.  Real data usually arrives **labelled** -- measurements by date,
prices by ticker symbol, populations by country -- and the labels are half the
information.

The `pandas` package provides two types for labelled data.  It is conventionally imported
as `pd`.

In [ ]:
import pandas as pd

prices = pd.Series([438.07, 451.67, 432.96, 431.41],
                   index=["Mon", "Tue", "Wed", "Thu"], name="Close")
prices

A **`Series`** is a numpy array together with an **index**, a list of labels for its
entries.  A value can be reached either way: by its label, or by its position.

In [ ]:
print(prices["Tue"])     # by label
print(prices.iloc[1])    # by position
print(prices.values)     # the bare numbers: a numpy array
print(prices.index)      # the labels

A **`DataFrame`** is a collection of `Series` sharing one index: a table, with a named
**column** per quantity and a labelled **row** per observation.

In [ ]:
week = pd.DataFrame({"Open":  [457.80, 447.99, 446.38, 435.90],
                     "Close": [438.07, 451.67, 432.96, 431.41]},
                    index=["Mon", "Tue", "Wed", "Thu"])
week

A single column is selected by name, as in `week["Close"]`, and the result is a `Series`.
Arithmetic on two columns is **aligned by the index**, not by position -- which is what
makes it safe to combine series whose entries are ordered differently, or missing.  A new
column is created simply by assigning to it.  The methods `idxmax` and `idxmin` then
report the **label** at which a series attains its largest or smallest value.

In [ ]:
week["Gain"] = week["Close"] - week["Open"]
print(week)
print("best day:", week["Gain"].idxmax())

Reading a real table from a file: `read_csv` takes the name of the column to use as the
index, and, with `parse_dates`, converts its entries into dates rather than leaving them
as strings.  The file `data/TSLA_ytd.csv` holds this year's daily prices of the Tesla
share, the same data as in Notebook 6, Task 4.

In [ ]:
tsla = pd.read_csv("data/TSLA_ytd.csv", index_col=0, parse_dates=True)
print(tsla.shape)
print(tsla.dtypes)
tsla.describe().round(2)

So the table has $172$ rows, one per trading day, and $5$ columns, four of them prices and
one a volume -- and `describe` summarizes each column separately.  Over the year the
closing price ranged from $298.32$ to $451.67$, with a mean of $390.75$.

---
**Exercises.**

1. What are the `type` and the `shape` of `tsla`, of `tsla["Close"]`, and of `tsla["Close"].values`?

2. The expression `tsla[["Open", "Close"]]` selects two columns at once.  Why the double brackets?  What is the `type` of the result?

3. On how many of the $172$ days did the closing price exceed the opening price?  (Hint: a comparison of two columns gives a `Series` of booleans, and `sum` counts the `True`s.)

4. Add a column `Range`, holding `High` minus `Low`, and find the date on which it was widest.

---

## Task 5: Time Series.

When the index of a `Series` consists of dates, it is a `DatetimeIndex`, and the `Series`
is a **time series**.  Slicing then accepts dates written as strings, and a partial date
selects a whole month or a whole year.

In [ ]:
close = tsla["Close"]
spring = close.loc["2026-03":"2026-04"]
print(len(spring), "trading days")
_ = spring.plot()

The plot labels its $x$-axis with the dates from the index, without our having to say so.

The **daily return** is the relative change from one closing price to the next,
$$
  r_t = \frac{S_t - S_{t-1}}{S_{t-1}},
$$
and `pct_change` computes it for the whole series at once.  The first value is `NaN`
("not a number"), as there is no previous day to compare with.

In [ ]:
ret = close.pct_change()
print(ret.head(3))
print("mean daily return:", ret.mean())

The mean daily return is $-0.00056$, that is, the share lost about $0.06\%$ per day on
average over this period.

The standard deviation of the daily returns measures how widely they scatter about that
mean.  It is the stock's **volatility**.  Multiplied by $\sqrt{252}$ -- there being about
$252$ trading days in a year -- it becomes the **annualized volatility**: the $\sigma$
that Notebook 9 will feed into the Black-Scholes formula.

In [ ]:
sigma = ret.std() * np.sqrt(252)
print(f"annualized volatility: {sigma:.1%}")

A **rolling window** applies a function to every window of `w` consecutive values.  With
`mean`, this is exactly the **moving average** of Notebook 6, Task 3 -- and with
`center=True`, it is the very same computation as correlation with the constant kernel
`np.ones(w)/w`.  Let's check that, with the `correlate_1d` function of Notebook 6.

In [ ]:
def correlate_1d(a, ker):            # as in Notebook 6, Task 2
    m, n = len(a), len(ker)
    a = np.pad(a, n//2)
    return np.array([(a[k:k+n] * ker).sum() for k in range(m)])

w = 15
roll = close.rolling(w, center=True).mean()
conv = correlate_1d(close.values, np.ones(w)/w)

inner = slice(w//2, len(close) - w//2)   # away from the edges
print(np.allclose(roll.values[inner], conv[inner]))

The two agree exactly, away from the edges.  At the edges they differ, and each is honest
about it in its own way: `rolling` reports `NaN` for the first and last `w//2` days, where
a full window is not available, while `correlate_1d` pads with zeros and so pulls the
average towards $0$ -- the falling ends we had to correct for in Notebook 6.

Any summary function can be used in a rolling window, not just `mean`.  Rolling the
standard deviation of the returns shows the volatility as it changed through the year.

In [ ]:
_ = (ret.rolling(30).std() * np.sqrt(252)).plot(
        title="TSLA: 30-day realized volatility, annualized")

Volatility is far from constant: it drifts between about $30\%$ and $70\%$ over the year,
peaking in late July.  This is why an option's **implied** volatility, computed in
Notebook 9 from a single market price, is a forecast rather than a measurement.

### A Light Curve.

Nothing here is particular to share prices: a time series is a time series.  Astronomers
watch a star's brightness over a night in exactly this way, and when a planet passes in
front of it, the brightness dips for a few hours.  This is how most known **exoplanets**
were found.

The file `data/tres2_transit.csv` holds eight hours of real measurements of the star
**TrES-2**, made by NASA's **Kepler** telescope in May 2009, one per minute.  The column
`hours` counts from the middle of the event, and `flux` is the star's brightness relative to
its normal level.  A planet, TrES-2 b, crosses the star during these eight hours.

In [ ]:
transit = pd.read_csv("data/tres2_transit.csv", index_col="hours")
light = transit["flux"]

print(len(light), "measurements")
_ = light.plot(style=".", ms=2, xlabel="hours from mid-transit")

There it is: for about an hour and a half the star is a little over one percent fainter, and either
side of that it sits at its normal brightness.  The shape repays a closer look.  The star
dims over some twenty-five minutes as the planet moves onto the disc (the **ingress**),
brightens again just as gradually at the **egress**, and in between the floor of the dip is not flat but
gently curved -- because a star is brighter at its centre than at its edge, an effect called
**limb darkening**.

The **depth** of the dip carries the physics.  A planet of radius $R_p$ crossing a star of
radius $R_s$ hides a fraction $(R_p/R_s)^2$ of the disc, so the ratio of the radii is the
square root of the depth.  We measure the depth by comparing the floor with the baseline,
selecting each with a boolean mask on the index.

In [ ]:
hours = light.index

baseline = light[abs(hours) > 1.5].mean()
floor    = light[abs(hours) < 0.4].mean()
depth    = baseline - floor

print(f"baseline   : {baseline:.6f}")
print(f"depth      : {depth:.5f}")
print(f"R_p / R_s  : {np.sqrt(depth):.3f}")

The planet's radius is $0.118$ times its star's.  TrES-2 is a star much like the Sun, and on
that assumption the planet comes out at about $1.2$ times the radius of Jupiter -- a gas
giant, orbiting its star every two and a half days.

Two remarks on the measurement.  The scatter of the baseline is $0.00023$, so a single
measurement pins the brightness to about $0.02\%$ and the transit stands out plainly; that
is Kepler's precision, and it is why the mission found planets far smaller than this one,
whose transits are shallower than the noise and emerge only after averaging over many
orbits.  And our depth is a slight underestimate, because limb darkening makes the dip
shallower than the simple ratio of areas would suggest.

---
**Exercises.**

1. Plot `close` together with its $15$-day and its $60$-day rolling mean, on one set of axes.  Which of the two follows the price more closely, and why?

2. The **log returns** are `np.log(close).diff()`.  Compute their annualized standard deviation and compare it with the $47.1\%$ above; why are the two close, but not equal?  Then use `close.shift(1)` to compute the ordinary daily returns by hand, and check them against `pct_change` with `np.allclose`.

3. On which date was the $30$-day realized volatility highest?  What was the share price doing around then?

4. Smooth the light curve with a $15$-minute rolling mean and plot it on top of the measurements, then recompute the depth from the smoothed curve.  By what factor did averaging reduce the scatter of the baseline, and is that the factor you expected?

5. The **duration** of a transit is usually quoted between the two moments at which the star is half way down to the floor.  Find those two times from the data, and hence the duration in hours.

---


## Task 6: Many Columns at Once.

The real convenience of a `DataFrame` shows when there are several series to compare.  The
file `data/prices_ytd.csv` holds this year's closing prices for four shares.

In [ ]:
stocks = pd.read_csv("data/prices_ytd.csv",
                     index_col=0, parse_dates=True)
stocks.head()

The four price series are on quite different scales, so plotting them together would say little
about their behaviour.  Dividing each column by its own first value **rebases** them all
to $1$, and then they can be compared.  Note that `stocks.iloc[0]` is the first *row*, a
`Series` indexed by the four ticker symbols, so the division is again aligned -- this time
column by column.

In [ ]:
_ = (stocks / stocks.iloc[0]).plot(
        title="growth of one euro invested on 2 January")

A method applied to a `DataFrame` acts on **every column**, and returns one value per
column.  So the annualized volatilities of all four shares come out of a single line.

In [ ]:
rets = stocks.pct_change()
(rets.std() * np.sqrt(252)).round(3)

Tesla, at $47\%$, is by some way the most volatile of the four, and Apple, at $27\%$, the
least.

The method `corr` goes one step further and compares the columns with **each other**: it
returns the matrix of **correlations** between the daily returns, a number between $-1$
and $1$ for each pair.  It is the basis of every portfolio computation, since a portfolio
of strongly correlated shares is barely diversified at all.

In [ ]:
rets.corr().round(2)

The diagonal is $1$, as every series correlates perfectly with itself, and the matrix is
symmetric.  Microsoft and Amazon move together most closely, at $0.41$, while Apple and
Amazon are almost unrelated, at $0.04$.

---
**Exercises.**

1. Which pair of the four shares moved together most closely?  Plot their rebased prices on one set of axes, to see it.

2. The expression `rets.groupby(rets.index.month).mean()` averages the daily returns by calendar month.  Which month was the best one for each of the four shares?

3. An equally weighted portfolio of the four has daily return `rets.mean(axis=1)`.  Compute its annualized volatility and compare it with the four individual figures.  Why is it lower than every one of them?

4. Using `yf.Ticker(...).history` as in Notebook 6, add a fifth share of your choice to the table, and repeat the correlation matrix.

---

## Summary.

This notebook introduced several core ideas about Python's treatment of functions, and a
first way to work with labelled data:

* **Functions as first-class objects:** they can be assigned to variables, stored in containers, and passed or returned like any other value.
* **Anonymous functions (lambdas):** compact ways to define small functions.
* **Closures:** a function defined inside another function carries along the variables it uses from it, which is what lets a function factory work, and what lets a function keep a cache of its own results.
* **General argument handling:** the use of `*args` and `**kwargs` to write functions that accept any number of positional or keyword arguments.
* **Labelled data:** a `pandas` `Series` pairs values with an **index**, and a `DataFrame` collects several such columns into a table; arithmetic then aligns by label rather than by position.
* **Time series:** a `DatetimeIndex` supports slicing by date, differencing into **returns**, and **rolling windows** -- the moving averages of Notebook 6, computed by `.rolling(w).mean()` instead of by convolution.
* **Summaries across columns:** a method applied to a `DataFrame` acts on every column at once, which turns volatilities and correlations into one-line computations.

Together, these ideas let us move from writing functions in the abstract to applying them to real data.